In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-02-25 12:14:41--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2025-02-25 12:14:41 (25.5 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with open('input.txt', 'r', encoding='utf-8') as ofile:
  text = ofile.read()
len(text)

1115394

In [3]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
# create a mapping from characters to integers.(createing encoder, decoder)
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for i,c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hii there'))
print(decode(encode('hii there')))
print(encode('hii iman'))
print(decode(encode('hii iman')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there
[46, 47, 47, 1, 47, 51, 39, 52]
hii iman


In [6]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [7]:
# split data to train and test
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
# we do not feed all training data to transformer, because it is expensive. we feed chunck of data with a predefine block size
block_size = 8
train_data[:block_size+1] # we add +1 to get 8 individual exampls of sequence: [1,2,3] has two individual sequence; 2 comes after 1, 3 comes after 1,2
# it means when we sample chunk of data like this, it has multiple examples packed into it. becuse all these charecters folow each others. and when we plug it
# to a transformer we are going simultaneously train it to make prediction at every one of these positions

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f'when input is {context} the target is {target}')

when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [10]:
# scince we are using GPUs and they are good in parallel processing. every time we give batch of training samples for more efficiency
# the batch will not see each other during training

In [11]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequence will be processed in parallel
block_size = 8 # what is the maximum context length for prediction?

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data)- block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1: i+block_size+1] for i in ix])
  return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets')
print(yb.shape)
print(yb)

print('--------------')

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b, :t+1]
    target = yb[b, t]
    print(f'when input is {context} the target is {target}')



inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
--------------
when input is tensor([24]) the target is 43
when input is tensor([24, 43]) the target is 58
when input is tensor([24, 43, 58]) the target is 5
when input is tensor([24, 43, 58,  5]) the target is 57
when input is tensor([24, 43, 58,  5, 57]) the target is 1
when input is tensor([24, 43, 58,  5, 57,  1]) the target is 46
when input is tensor([24, 43, 58,  5, 57,  1, 46]) the target is 43
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target is 39
when input is tensor([44]) the target is 53
when input is tensor([44, 53]) the target is 56
when input is tensor([44, 53, 56]) the 

In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)# create an embedding of size vocab_size for each vocab

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C) # take embedding from token_embedding_table correspods to each indices in idx and consider it as next token related to that indice

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # cross_entropy function accept shape of (N, C) where N = B*T (total tokens in the batch)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
            #print(idx, idx.shape)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=10)[0].tolist()))


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

Sr?qP-QWkt


In [13]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [14]:
batch_size = 32
for steps in range(10000):  # increase number of steps for good results...
  # sample a batch of data
  xb,yb = get_batch('train')
   # evaluate the loss
  logits, loss = m(xb,yb)

  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())


2.4730143547058105


In [15]:
# after doing optimization on logits(10000 iteration and each time we picked 4 batch of 8 samples) to tune the embeddings and get the tuned embeddings
# then calling the generate method to see the new result
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


By y a!
Caril.n y, denjxhece w illd CHAL, mer thoun s's:Conchuntilalllevise sthat dy hangilyoteng h 


## The mathematical trick in self-attention
- we want to some how use all information exist preceeding the current time stamp

In [16]:
## Consider this example
torch.manual_seed(1982)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

### First method

In [17]:
# we want each time step has the information of previous steps. by getting average of embedding of previous steps up to current time step
xbow = torch.zeros((B,T,C))# creating bag of words with the same dimenssion as x
for b in range(B): # in each bach
  for t in range(T):  # for each time step
    temp = x[b,:t+1] # extract embeddings of previous steps up to current step
    xbow[b,t] = torch.mean(temp, 0) # get the avg of embeddings and assign it to the value of current time step



In [18]:
x[0]

tensor([[-1.1130,  0.4103],
        [ 0.2952, -0.0519],
        [-1.6193, -0.3191],
        [-0.8132,  0.5285],
        [-0.1648, -0.5005],
        [-1.3201,  1.0941],
        [-2.4110, -1.1813],
        [-1.1350, -0.4219]])

In [19]:
xbow[0]

tensor([[-1.1130,  0.4103],
        [-0.4089,  0.1792],
        [-0.8124,  0.0131],
        [-0.8126,  0.1420],
        [-0.6830,  0.0135],
        [-0.7892,  0.1936],
        [-1.0209, -0.0028],
        [-1.0352, -0.0552]])

### Second method. by matrix multiplication

In [20]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
print('Triangle matrix:')
print(a)


print('----------------')
a  = a/torch.sum(a, dim = 1, keepdim=True)
print(a)


print('----------------')
b = torch.randint(0,10, (3,2)).float()
print(b)


print('----------------')
c = a@b
print(c)




Triangle matrix:
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
----------------
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
----------------
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
----------------
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [21]:
# now lets try matrix multiplication to get average of embedings
wei = torch.tril(torch.ones(T,T)) # we want to calculate avg on a T,C matrix so the triangular lower matrix should be T,T
wei = wei/torch.sum(wei, dim = 1, keepdim=True)

xbow2 = wei @ x  # x is B,T,C but wei is T,T => torch change wei to B,T,T meaning multiply wei to each bach
xbow2.shape


torch.Size([4, 8, 2])

In [22]:
torch.allclose(xbow,xbow2)

True

In [23]:
xbow[0], xbow2[0]

(tensor([[-1.1130,  0.4103],
         [-0.4089,  0.1792],
         [-0.8124,  0.0131],
         [-0.8126,  0.1420],
         [-0.6830,  0.0135],
         [-0.7892,  0.1936],
         [-1.0209, -0.0028],
         [-1.0352, -0.0552]]),
 tensor([[-1.1130,  0.4103],
         [-0.4089,  0.1792],
         [-0.8124,  0.0131],
         [-0.8126,  0.1420],
         [-0.6830,  0.0135],
         [-0.7892,  0.1936],
         [-1.0209, -0.0028],
         [-1.0352, -0.0552]]))

### Third method. Using softmax

In [24]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow2,xbow3)

True

#### Forth method: Self-Attention

In [25]:
torch.manual_seed(42)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16

query = nn.Linear(C, head_size, bias = False)
key = nn.Linear(C, head_size, bias = False)
value = nn.Linear(C, head_size, bias = False)

q = query(x) # B,T,head_size
k = key(x)   # B,T,head_size
v = value(x) # B,T,head_size

wei = q @ k.transpose(-2,-1)   #(B, T, head_size) @ (B, head_size, T) ----> (B, T, T)

tri = torch.tril(torch.ones(T,T))
wei = tri @ wei

out = wei @ v

out.shape

# Query (Q): Represents the current token’s intent—what it is looking for in other tokens.
# Key (K): Represents the information each token offers—how relevant it is to other tokens.
# Value (V): Contains the actual content/information of each token to be aggregated based on attention scores.

torch.Size([4, 8, 16])

- 1. Query (Q) and Key (K)
  - Q: Represents what the token is looking for.
  - K: Represents what the token offers(what I have as a token).
  - Dot product (q @ k.T) computes attention scores, showing how much each token relates to every other token.
- 2. Value (V)
  - If you find me interesting here is what I communicate to you
  - Stores actual information that is retrieved based on attention weights.
  - Unlike static embeddings, V is dynamic and changes based on context.
  - out = wei @ v gives the final attended representation of each token.
-3. Attention Heads
  - Instead of using one big head, we use multiple smaller heads.
Each head learns different relationships (e.g., short-term vs. long-term dependencies).
  - Why reduce embedding size?
We split the original embedding size into smaller head sizes to distribute learning across multiple heads.

In [26]:
wei[0]

tensor([[-0.3332, -0.6597,  0.3630, -0.1001,  0.0136, -0.5833,  1.1351, -1.2784],
        [-1.5055,  0.1272, -1.1589,  0.7648, -1.6066,  0.6191, -0.8589, -1.7338],
        [-2.5271, -1.1454, -0.3768,  0.7313, -3.5954,  0.2910,  0.6956, -3.1456],
        [-2.5816,  0.5398, -2.0983,  1.7534, -3.9281,  1.2056, -1.1081, -2.5064],
        [-3.6766,  0.6557, -2.4478,  1.6184, -5.1788,  2.1865, -1.6143, -3.0844],
        [-3.4031,  1.2008, -2.1593,  1.3107, -6.0716,  1.7006, -4.2252, -1.1553],
        [-3.2690,  1.4363, -2.2614,  1.4547, -8.3390,  3.4595, -5.2991,  0.5136],
        [-4.1180,  1.2402, -3.6885,  1.1528, -5.2829,  3.6245, -3.6561,  0.6239]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below